In [21]:
import weaviate
import os
from dotenv import load_dotenv

load_dotenv(override=True)
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.getenv("WEAVIATE_URL"),
    auth_credentials=weaviate.auth.AuthApiKey(os.getenv("WEAVIATE_API_KEY")),
    headers={"X-Cohere-Api-Key": os.getenv("COHERE_API_KEY")}
)

if client.is_ready():
    print("✅ СЕГА ВЕЧЕ СИ СВЪРЗАН УСПЕШНО!")

✅ СЕГА ВЕЧЕ СИ СВЪРЗАН УСПЕШНО!


In [22]:
import weaviate.classes as wvc

client.collections.delete("Product")

client.collections.create(
    name="Product",
    vectorizer_config=wvc.config.Configure.Vectorizer.text2vec_cohere(),
    generative_config=wvc.config.Configure.Generative.cohere(),
    properties=[
        wvc.config.Property(name="name", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="price", data_type=wvc.config.DataType.NUMBER),
    ]
)

products_coll = client.collections.get("Product")
products_coll.data.insert_many([
    {"name": "Laptop X1", "price": 1200},
    {"name": "Gaming Mouse", "price": 50},
    {"name": "Wireless Headphones", "price": 150}
])

print("✅ Колекцията 'Product' вече е ИНТЕЛИГЕНТНА и готова!")

✅ Колекцията 'Product' вече е ИНТЕЛИГЕНТНА и готова!


In [1]:
products = client.collections.get("Product")

questions = [
    "Какви продукти предлагате?",
    "Кой е най-скъпият продукт?",
    "Имате ли нещо за слушане на музика?",
    "А нещо за геймъри?",
    "Предложи ми евтин продукт под 100 лева."
]

print("🚀 Асистентът започва да отговаря...")

for q in questions:
    print(f"\n❓ Въпрос: {q}")

    search_res = products.query.near_text(query=q, limit=5)

    items_info = ""
    for obj in search_res.objects:
        items_info += f"- {obj.properties['name']}: {obj.properties['price']} лв.\n"

    prompt = f"""
    Ти си любезен асистент. Твоята задача е да отговаряш ВИНАГИ на български.
    Използвай този списък с продукти:
    {items_info}

    Отговори на въпроса на клиента: {q}
    Ако има продукт под 100 лв (като мишката), задължително го спомени!
    """

    res = products.generate.near_text(
        query=q,
        single_prompt=prompt,
        limit=1
    )

    if res.objects:
        print(f"🤖 Асистент: {res.objects[0].generated}")

NameError: name 'client' is not defined